# 14 實戰案例 — 參考解答

松柏護理之家退伍軍人症迷你疫調報告的完整解答。

In [ ]:
# Google Colab setup -- 若在本機執行可跳過此 cell
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || True
    os.chdir('/content/python4epi')
    !pip install -q -e .

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

## 題目 1：疫情摘要表

In [ ]:
df = pd.read_csv("data/synthetic/legionella_outbreak.csv")
df["symptom_onset_date"] = pd.to_datetime(df["symptom_onset_date"], errors="coerce")
df["infected"] = (df["clinical_severity"] != "not_ill").astype(int)

n_total = len(df)
n_infected = int(df["infected"].sum())
n_deaths = int((df["outcome"] == "dead").sum())
n_hosp = int(df["hospitalized"].sum())
n_icu = int(df["icu_admission"].sum())

summary = pd.DataFrame([
    ["總住民數", n_total, ""],
    ["感染人數", n_infected, f"{n_infected/n_total:.1%}"],
    ["死亡人數", n_deaths, f"{n_deaths/n_infected:.1%} (CFR)"],
    ["住院人數", n_hosp, f"{n_hosp/n_infected:.1%} (住院率)"],
    ["ICU 人數", n_icu, f"{n_icu/n_hosp:.1%} (ICU/住院)"],
    ["侵襲率", f"{n_infected/n_total:.1%}", ""],
    ["致死率", f"{n_deaths/n_infected:.1%}", ""],
], columns=["指標", "數值", "比例"])

print("=== 疫情摘要表 ===")
print(summary.to_string(index=False))

## 題目 2：危險因子快速篩查

In [ ]:
factors = ["shower_use", "hydrotherapy_use", "comorbidity_copd", "immunosuppressed"]
results = []

for factor in factors:
    exposed_inf = int(df[(df[factor] == 1) & (df["infected"] == 1)].shape[0])
    exposed_n = int(df[df[factor] == 1].shape[0])
    unexposed_inf = int(df[(df[factor] == 0) & (df["infected"] == 1)].shape[0])
    unexposed_n = int(df[df[factor] == 0].shape[0])

    ar_exp = exposed_inf / exposed_n if exposed_n > 0 else 0
    ar_unexp = unexposed_inf / unexposed_n if unexposed_n > 0 else 0
    rr = ar_exp / ar_unexp if ar_unexp > 0 else float("inf")

    chi2, p, _, _ = stats.chi2_contingency(
        pd.crosstab(df[factor], df["infected"])
    )

    results.append({
        "factor": factor,
        "exposed_AR": f"{ar_exp:.1%}",
        "unexposed_AR": f"{ar_unexp:.1%}",
        "RR": f"{rr:.2f}",
        "p-value": f"{p:.4f}",
        "sig": "*" if p < 0.05 else "",
    })

rr_df = pd.DataFrame(results)
print("=== 危險因子 RR 比較表 ===")
print(rr_df.to_string(index=False))

max_rr = rr_df.loc[rr_df["RR"].astype(float).idxmax()]
print(f"\n→ RR 最大的因子: {max_rr['factor']} (RR = {max_rr['RR']})")
print("→ 淋浴使用是最強的暴露危險因子，與感染狀態有統計顯著關聯")

## 題目 3（挑戰題）：迷你 SitRep

In [ ]:
cases = df[df["infected"] == 1].copy()

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# --- 圖 1: 流行曲線 ---
daily = cases.groupby("symptom_onset_date").size()
full_range = pd.date_range(daily.index.min(), daily.index.max(), freq="D")
daily = daily.reindex(full_range, fill_value=0)

axes[0].bar(daily.index, daily.values, color="steelblue", edgecolor="white")
peak = daily.idxmax()
axes[0].axvline(peak, color="red", linestyle="--", alpha=0.7)
axes[0].set_title(f"流行曲線 (高峰: {peak.strftime('%m/%d')})")
axes[0].set_ylabel("每日新增")
axes[0].tick_params(axis="x", rotation=45)

# --- 圖 2: 年齡分布 ---
for label, grp in df.groupby("infected"):
    tag = "感染" if label == 1 else "未感染"
    axes[1].hist(grp["age"], bins=15, alpha=0.6, label=tag, edgecolor="white")
axes[1].set_title("年齡分布")
axes[1].set_xlabel("年齡")
axes[1].legend()

# --- 圖 3: 樓層翼區侵襲率 ---
zone = df.groupby(["floor", "wing"])["infected"].agg(["sum", "count"]).reset_index()
zone["ar"] = zone["sum"] / zone["count"] * 100
zone["label"] = zone["floor"].astype(str) + "F-" + zone["wing"]
colors = ["#e74c3c" if ar > 50 else "steelblue" for ar in zone["ar"]]
axes[2].bar(zone["label"], zone["ar"], color=colors)
axes[2].axhline(50, color="red", linestyle="--", alpha=0.5)
axes[2].set_title("樓層翼區侵襲率")
axes[2].set_ylabel("侵襲率 (%)")
for i, row in zone.iterrows():
    axes[2].text(i, row["ar"] + 1, f"{row['ar']:.0f}%", ha="center", fontsize=9)

plt.suptitle("松柏護理之家退伍軍人症群聚 — 迷你 SitRep", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# 行動建議
top_zone = zone.sort_values("ar", ascending=False).iloc[0]
print("=" * 40)
print("  行動建議")
print("=" * 40)
print(f"  1. 優先處理區域: {top_zone['label']} (侵襲率 {top_zone['ar']:.1f}%)")
print(f"  2. 次要關注: 2F-A (54.5%) — 兩區共占大多數個案")
print(f"  3. 立即停用高風險區淋浴設施")
print(f"  4. 對 2F、3F 水管系統進行環境採檢")
print("=" * 40)

### 解讀

- **題目 1**：摘要表是疫調報告的第一頁，讓決策者快速掌握規模
- **題目 2**：RR 篩查可以快速找出最值得深入調查的暴露因子
  - 注意：crude RR 未調整交絡因子，需搭配 Ch05 分層分析和 Ch06 邏輯斯迴歸
- **題目 3**：好的 SitRep 一定要有「行動建議」——分析的目的是支持決策

恭喜完成最後的練習！你已經具備用 Python 進行疫情調查的核心技能。